# Core PyTorch Mechanics: Autograd, Modules, and Training


## 1. Autograd and the Computational Graph

__What is a leaf tensor?__

A leaf tensor is a tensor at the beginning of the computational graph. It was not created by an operation tracked by autograd. By default, any tensor we create (e.g. `torch.tensor([1., 2.], requires_grad=True)` is a leaf. Parameters of neural networks (`nn.Parameter`) are also leaf tensors. Non-leaf tensors are the results of operations (`y = x + 2`).

__When does `.grad` populate?__

The `.grad` attribute for leaf tensors when you call `.backward()` on a scalar output (like loss) that depends on those leaves. By default, PyTorch frees the graph after `.backward()` and non-leaf tensors do not have their gradients saved to memory.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Creating Tensors

In [7]:
import numpy as np

# from python/numpy data
a = torch.tensor([1., 2., 3.])
b = torch.tensor(np.array([2., 3., 4.]))
c = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)

# factory functions
zeros = torch.zeros(3, 4)   # shape is NOT a tuple
ones = torch.ones(3, 4, dtype=torch.float32)    # default for float is float32 due to GPU compatibility
rng = torch.rand(5, 5)      # standard normal
eye = torch.eye(3)          # identity


# from numpy
np_arr = np.arange(1, 5, dtype=float)
t_arr = torch.from_numpy(np_arr)        # shares memory w/ np_arr
t_arr2 = torch.tensor(np_arr)           # DOES NOT share memory w/ np_arr

t_arr[0] = 1000.

print(np_arr)

[1000.    2.    3.    4.]


### Data Types (`dtypes`)

```python
torch.float32   # default in torch
torch.float64
torch.float16
torch.bfloat16
torch.int32
torch.int64     # default in torch
torch.bool
```

In [8]:
x = torch.tensor([1, 2, 3])
print(x.dtype)

y = x.float()   # casts to float32
z = x.to(torch.float16) # explicit cast

torch.int64


### Shape, Stride, and Storage

A tensor's __storage__ is a contiguous, one-dimensional array of numbers in memory. Multiple tensors can share the same storage.

The __stride__ of a tensor is a tuple of integers. The _k_-th stride tells you how many elements you must skip to move one position along dimension _k_.

Given a $m \times n$ tensor, the element at position (i, j) is at storage offset:
    $offset = i * stride[0] + j*stride[1]$

To get to the next row's element (at same "place") need to move `stride[0]` places. Note that in a `mxn` matrix, `stride[0] = tensor.shape[-1]` - the number of columns.

In [12]:
x = torch.arange(1, 7, dtype=torch.float).reshape(2, 3)

print(x.shape)
print(x.stride())   # (3, 1)
# Moving along dim 0 (row): skip 3 elements
# Moving along dim 1 (col): skip 1 element


torch.Size([2, 3])
(3, 1)


# Views vs. Copies

Many PyTorch operations return __views__: tensors that share the same underlying storage but with different shape/stride metadata.

In [15]:
a = torch.arange(12)
print(f"{a=}")

b = a.view(3, 4)            # view: shared storage
print(f"{b=}")

c = a.reshape(3, 4)          # view if possible, copy o.w.

b[0, 0] = 99

print(f"{a[0]=}")
print(f"{b[0]=}")
print(f"{c[0]=}")

# Transpose returns a view (non-contiguous)
t = torch.randn(3, 4)
print(f"{t=}")

t_T = t.T
print(f"{t_T=}")

print("\n")
print(f"{t.stride()=}")
print(f"{t_T.stride()=}")
print(f"{t_T.is_contiguous()=}")

a=tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
b=tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
a[0]=tensor(99)
b[0]=tensor([99,  1,  2,  3])
c[0]=tensor([99,  1,  2,  3])
t=tensor([[ 0.6781, -0.1067, -1.3776,  0.9754],
        [-0.0489, -0.7441,  0.1770, -1.1156],
        [ 0.2698, -1.1076,  0.3067,  0.9394]])
t_T=tensor([[ 0.6781, -0.0489,  0.2698],
        [-0.1067, -0.7441, -1.1076],
        [-1.3776,  0.1770,  0.3067],
        [ 0.9754, -1.1156,  0.9394]])


t.stride()=(4, 1)
t_T.stride()=(1, 4)
t_T.is_contiguous()=False
